# Debate vs supervisor vs swarm [Step 06.03 - Three different reasons to have many agents]

> **MLCourse - Agentic AI - Agent Patterns**

You have already built two multi-agent architectures in
[`../../02_langgraph/06_multi_agent_systems`](../../02_langgraph/06_multi_agent_systems):

- `01_supervisor_pattern.ipynb` - a supervisor agent that **routes work to
  specialists** and collects results.
- `02_swarm_handoff.ipynb` - peers that **hand control to each other** when the
  conversation moves into someone else's expertise.

Debate is a third thing, and the difference is not architectural detail - it is
*what the extra agents are for*.

| | **Supervisor** | **Swarm** | **Debate** |
|---|---|---|---|
| Agents differ by | **capability** (tools, domain) | **capability** (tools, domain) | **stance** (same capability) |
| Work is | **divided** | **passed along** | **duplicated on purpose** |
| Disagreement is | a bug | rare - only one agent is active | **the entire point** |
| Control flow | central router, fan-out/fan-in | decentralised handoff | parallel, then a judge |
| Cost scales with | number of subtasks | conversation length | rounds x agents (quadratic-ish in context) |
| Buys you | **coverage** and tool access | **continuity** across domains | **error correction** |
| Fails when | the router mis-assigns | nobody hands off (or ping-pong) | agents agree, or fold |

### The one-sentence version

- Supervisor and swarm exist because **one agent cannot do all the work**.
- Debate exists because **one agent cannot check its own work**.

If your problem is "this needs a SQL tool and a web tool and a writer", debate is
the wrong pattern - it will run the same incapable agent twice. If your problem is
"the answer is sometimes confidently wrong and I can't tell which times", supervisor
and swarm do nothing for you, because a specialist is just as capable of being
confidently wrong within its specialty.

### Key takeaways

- Choose by **failure mode**, not by architecture diagram.
- The patterns compose: a supervisor may route one hard subtask into a debate.
- Debate is the only one of the three whose value you can measure with a grader,
  which is why this module can honestly tell you when it does not work.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.5
print("PACE =", PACE)

PACE = 2.5


### The task set


In [ ]:
# Four questions with a single, checkable numeric answer. They are deliberately of
# the "first instinct is wrong" family: the point of debate is supposed to be that
# a second agent catches what the first one missed.
#
# A DETERMINISTIC grader matters more than the questions. If an LLM grades, you are
# measuring the grader as much as the method.

import re

TASKS = [
    dict(id="bat_ball",
         q="A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
           "How much does the ball cost, in dollars?",
         answer=0.05),
    dict(id="widgets",
         q="If 5 machines take 5 minutes to make 5 widgets, how many minutes would "
           "100 machines take to make 100 widgets?",
         answer=5),
    dict(id="strawberry",
         q="How many times does the letter 'r' appear in the word 'strawberry'?",
         answer=3),
    dict(id="avg_speed",
         q="A car drives 60 km at 30 km/h, then another 60 km at 60 km/h. "
           "What is its average speed over the whole trip, in km/h?",
         answer=40),
]

NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")


def grade(text, expected, tol=1e-6):
    """Deterministic grader: take the LAST number in the reply and compare.

    Every method in this module is told to end with 'FINAL: <number>', so the last
    number is the stated answer. No LLM opinion is involved anywhere in scoring.
    """
    nums = NUM_RE.findall((text or "").replace(",", "").replace("$", ""))
    if not nums:
        return False, None
    got = float(nums[-1])
    return abs(got - expected) <= max(tol, abs(expected) * 1e-6), got


ANSWER_RULE = ("Think briefly, then end your reply with a line of exactly the form "
               "'FINAL: <number>' and nothing after it.")

print("%d tasks" % len(TASKS))
for t in TASKS:
    print("  %-11s expected %s" % (t["id"], t["answer"]))


### 1. The same problem, two shapes

Take a question that needs two *different* things done - a lookup and a calculation.
We solve it once with a supervisor and once with a debate, and look at what each
architecture actually did.

In [4]:
# A tiny world with facts an agent cannot know, so "capability" is real here.
CATALOGUE = {"widget": 12.50, "gizmo": 4.25, "doohickey": 31.00}
VAT_RATE = 0.21

COMPOSITE_Q = ("A customer buys 3 widgets and 2 gizmos. Using the catalogue prices, "
               "what is the total including VAT, in euros?")

TRUE_TOTAL = round((3 * CATALOGUE["widget"] + 2 * CATALOGUE["gizmo"]) * (1 + VAT_RATE), 2)
print("ground truth: EUR", TRUE_TOTAL)

ground truth: EUR 55.66


### SUPERVISOR shape: divide the work by capability


In [ ]:
# The "lookup specialist" has the catalogue; the "maths specialist" does not, and
# does not need it. The supervisor composes them. Note: no disagreement is possible,
# because the two agents are never asked the same question.

lookup_llm = make_llm(temperature=0.0, max_tokens=180)
maths_llm = make_llm(temperature=0.0, max_tokens=220)

sup_calls = []


def lookup_specialist(question):
    facts = "\n".join("%s = EUR %.2f" % (k, v) for k, v in CATALOGUE.items())
    m = safe_invoke(lookup_llm, [
        ("system", "You are the catalogue specialist. You have the price list below. "
                   "Extract ONLY the line items and unit prices the question needs. "
                   "Do not compute a total.\n\nPRICE LIST:\n" + facts),
        ("user", question)])
    sup_calls.append(m)
    return m.content.strip()


def maths_specialist(question, facts):
    m = safe_invoke(maths_llm, [
        ("system", "You are the arithmetic specialist. You are given facts by a "
                   "colleague; trust them. VAT is %.0f%%." % (VAT_RATE * 100)),
        ("user", "%s\n\nFACTS FROM THE CATALOGUE SPECIALIST:\n%s\n\n%s"
                 % (question, facts, ANSWER_RULE))])
    sup_calls.append(m)
    return m.content.strip()


print("[supervisor] -> lookup_specialist")
facts = lookup_specialist(COMPOSITE_Q)
print(facts)
print()
print("[supervisor] -> maths_specialist")
sup_answer = maths_specialist(COMPOSITE_Q, facts)
print(sup_answer)

sup_ok, sup_num = grade(sup_answer, TRUE_TOTAL, tol=0.02)
sup_tok = sum((m.usage_metadata or {}).get("total_tokens", 0) for m in sup_calls)
print()
print("supervisor: %s (got %s, truth %s), %d calls, %d tokens"
      % ("CORRECT" if sup_ok else "WRONG", sup_num, TRUE_TOTAL, len(sup_calls), sup_tok))


### DEBATE shape: duplicate the work and argue


In [ ]:
# Both agents get the SAME question and the SAME catalogue. Nothing is divided.

facts_block = "\n".join("%s = EUR %.2f" % (k, v) for k, v in CATALOGUE.items())
CONTEXT = ("PRICE LIST:\n%s\nVAT is %.0f%%." % (facts_block, VAT_RATE * 100))

deb_a = make_llm(temperature=0.0, max_tokens=250)
deb_b = make_llm(temperature=0.0, max_tokens=250)
deb_calls = []

A_SYS = "You are Agent A. Solve directly and concisely." + " " + CONTEXT
B_SYS = ("You are Agent B, a sceptic. Check the arithmetic and whether VAT was "
         "applied to the right base. " + CONTEXT)

a1 = safe_invoke(deb_a, [("system", A_SYS), ("user", COMPOSITE_Q + " " + ANSWER_RULE)])
b1 = safe_invoke(deb_b, [("system", B_SYS), ("user", COMPOSITE_Q + " " + ANSWER_RULE)])
deb_calls += [a1, b1]
print("[A]", a1.content.strip()[:300])
print()
print("[B]", b1.content.strip()[:300])

judge_llm = make_llm(temperature=0.0, max_tokens=200)
jm = safe_invoke(judge_llm, [
    ("system", "You are an impartial judge. Decide which position is correct. "
               "Ignore length and confidence. One sentence, then 'VERDICT: A' or "
               "'VERDICT: B', then 'FINAL: <number>'." + " " + CONTEXT),
    ("user", "QUESTION:\n%s\n\nPOSITION A:\n%s\n\nPOSITION B:\n%s"
             % (COMPOSITE_Q, a1.content, b1.content))])
deb_calls.append(jm)
print()
print("[judge]", jm.content.strip()[:300])

deb_ok, deb_num = grade(jm.content, TRUE_TOTAL, tol=0.02)
deb_tok = sum((m.usage_metadata or {}).get("total_tokens", 0) for m in deb_calls)
print()
print("debate: %s (got %s, truth %s), %d calls, %d tokens"
      % ("CORRECT" if deb_ok else "WRONG", deb_num, TRUE_TOTAL, len(deb_calls), deb_tok))


In [7]:
print("=" * 64)
print("%-14s %-9s %7s %9s   %s" % ("pattern", "result", "calls", "tokens", "what it bought"))
print("-" * 64)
print("%-14s %-9s %7d %9d   %s" % ("supervisor", "CORRECT" if sup_ok else "WRONG",
                                   len(sup_calls), sup_tok, "access to the catalogue"))
print("%-14s %-9s %7d %9d   %s" % ("debate", "CORRECT" if deb_ok else "WRONG",
                                   len(deb_calls), deb_tok, "a second opinion"))
print("=" * 64)

pattern        result      calls    tokens   what it bought
----------------------------------------------------------------
supervisor     CORRECT         2       461   access to the catalogue
debate         CORRECT         3      1334   a second opinion


### The point of that comparison

Both may well get it right - this is not a hard sum. What differs is **which risk
each one removes**:

- The supervisor removes the risk that the agent does not have the catalogue. If we
  delete the catalogue from the debate's context, the debate fails and *no amount of
  arguing recovers it* - two agents confidently hallucinating prices produce a
  confident hallucinated total.
- The debate removes the risk that VAT is applied to the wrong base. A supervisor
  cannot catch that, because nobody in the supervisor architecture is asked to
  disagree with the maths specialist.

Different failure modes. Different patterns. Neither is "more advanced".

### 2. Swarm, and why it is not in this comparison

The swarm pattern's job is **continuity**: a single conversation drifts from
billing into technical support, and control hands over without the user restarting.
Its unit of work is a *conversation*, not a *question*. There is nothing for a judge
to adjudicate, because only one agent is ever active.

Swarm and debate are close to orthogonal. You can have both: each swarm member may
internally run a debate before answering, and that is a reasonable design if that
member's answers are high-stakes.

### 3. Choosing

Ask, in this order:

1. **Does one agent lack the tools or knowledge to finish?** -> supervisor
   (or hierarchical supervisors if the subtask tree is deep).
2. **Does the conversation span domains, with the user staying put?** -> swarm.
3. **Is the answer sometimes confidently wrong, and can you grade it?** -> debate,
   *and only if you can grade it*, because otherwise you cannot know it helped.
4. **None of the above?** -> a single agent. Multi-agent architectures are a cost
   and a source of new failure modes; they need a reason.

### Next

Notebook 04 does the honest arithmetic: single pass vs self-consistency vs debate
on the same four graded tasks, with accuracy and cost side by side.